In [3]:
%pip install pandas sqlalchemy mysql-connector-python

  Using cached pandas-3.0.6-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached sqlalchemy-2.1.3-cp314-cp314-win_amd64.whl.metadata (9.9 kB)
  Using cached mysql_connector_python-26.7.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
Using cached pandas-3.0.6-cp314-cp314-win_amd64.whl (9.8 MB)
Using cached sqlalchemy-2.1.3-cp314-cp314-win_amd64.whl (2.4 MB)
Using cached mysql_connector_python-26.7.0-cp314-cp314-win_amd64.whl (18.2 MB)
Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl (12.7 MB)

   ---------------------------------------- 0/5 [tzdata]
   ---------------------------------------- 0/5 [tzdata]
   ---------------------------------------- 0/5 [tzdata]
   ---------------------------------------- 0/5 [tzdata]
   -------- ------------------------------- 1/5 [sqlalchemy]
   -------- ------------------------------- 1/5 [sqlalchemy]
   -------- ------------------------------- 1/5 [sqlalchemy]
   -------- ----

In [4]:
import pandas as pd
import logging
import sys
import os
from sqlalchemy import create_engine

In [5]:
db_host="localhost",
db_user="root",       
db_password="password",
db_name="e-commerce"


engine = create_engine(f"mysql+mysqlconnector://{db_user}:{db_password}@{db_host}/{db_name}")

In [6]:
logging.basicConfig(
    filename='etl_log.log', 
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [7]:
def extract_data(csv_files, json_files):
    csv_data = {} 
    json_data = {}

    # CSV files
    for file in csv_files:
        table_name = file.split('.')[0]
        csv_data[table_name] = pd.read_csv(file)

    # JSON files
    for file in json_files:
        table_name = file.split('.')[0]
        json_data[table_name] = pd.read_json(file)

    return csv_data, json_data


In [8]:
def load_data(df_csv, df_json):
    try:
        with engine.connect() as conn:
            for table_name, df in {**df_csv, **df_json}.items():
                df.to_sql(table_name, conn, if_exists="replace", index=False)
                logging.info(f"✅ Loaded {table_name} into database.")
    except Exception as e:
        logging.error(f"❌ Error loading data: {e}")

In [12]:
def main():
    logging.info("ETL process started.")

    csv_files = ["orders.csv", "order_items.csv", "website_pageviews.csv", "website_sessions.csv"]
    json_files = ["order_item_refunds.json", "products.json"]

    df_csv, df_json = extract_data(csv_files, json_files)
    load_data(df_csv, df_json)

    logging.info("ETL process completed successfully.")

if __name__ == "__main__":
    main()

In [13]:
if __name__ == "__main__":
    main()